<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

# Regression - House Price Prediction

## Comparison of Loss Functions

#### Importing necessary modules

In [ ]:
#Importing Modules
import sklearn
import scipy
import scipy.stats as stats
from scipy.stats import skew,boxcox_normmax, zscore
from scipy.special import boxcox1p
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer,KNNImputer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
# Re-load data and apply previous preprocessing steps
train_data = pd.read_csv("https://raw.githubusercontent.com/vigneashpandiyan/Pytorch-for-KTEK0070-Machine-Learning-in-Digital-Manufacturing-/main/Pytorch%20DL/Data/House%20price/train.csv", index_col="Id")
test_data = pd.read_csv("https://raw.githubusercontent.com/vigneashpandiyan/Pytorch-for-KTEK0070-Machine-Learning-in-Digital-Manufacturing-/main/Pytorch%20DL/Data/House%20price/test.csv", index_col="Id")

# Remove outliers (from cell FrSpIQ5qaERH)
train_data = train_data[train_data["GrLivArea"]<4450]
# Merging train and test data (from cell FrSpIQ5qaERH)
data = pd.concat([train_data.drop("SalePrice",axis=1),test_data])


In [ ]:
y1 = train_data['SalePrice']

# Plot 1: Normal Distribution
plt.figure(2, figsize=(8, 6)) # Explicitly set figure number and size
ax_normal = plt.gca() # Get current axes
ax_normal.set_title('Normal')
sns.histplot(y1, kde=True, stat='density', ax=ax_normal, color='skyblue')
ax_normal.ticklabel_format(style='sci', scilimits=(0,0), axis='y') # Scientific notation for y-axis
ax_normal.grid(False) # No grid
ax_normal.set_facecolor('white') # White background

# Plot 2: Log Normal Distribution
plt.figure(3, figsize=(8, 6)) # Explicitly set figure number and size
ax_lognormal = plt.gca() # Get current axes
ax_lognormal.set_title('Log Normal')
sns.histplot(y1, kde=True, stat='density', ax=ax_lognormal, color='lightcoral') # Plotting y1 as per original cell
ax_lognormal.ticklabel_format(style='sci', scilimits=(0,0), axis='y') # Scientific notation for y-axis
ax_lognormal.grid(False) # No grid
ax_lognormal.set_facecolor('white') # White background

plt.tight_layout() # Adjust layout to prevent overlapping
plt.show() # Display plots


In [ ]:

# Impute categorical features with "None" (from cell BBVbxV3caERN)
features_nonefill = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", "GarageCond", "GarageQual", "GarageFinish", "GarageType", "BsmtCond", "BsmtExposure", "BsmtQual", "BsmtFinType2", "BsmtFinType1"]
data[features_nonefill] = data[features_nonefill].fillna("None")

# Impute categorical features with "Mode" (from cell vk3K_gE-aERN)
pd.set_option('future.no_silent_downcasting', True)
features_modefill = ["MasVnrType", "MSZoning", "Utilities", "Exterior1st", "Exterior2nd", "SaleType", "Electrical", "KitchenQual", "Functional"]
data[features_modefill] = data.groupby("Neighborhood")[features_modefill].transform(lambda x:x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))

# Impute numerical features with "median" (from cell fV8sMyPcaERN)
features_medianfill = ["GarageArea", "LotFrontage"]
data[features_medianfill] = data.groupby("Neighborhood")[features_medianfill].transform(lambda x: x.fillna(x.median()))

# Impute numerical features with 0 (from cell d0HXgasFaERO)
features_zerofill = ["GarageYrBlt", "MasVnrArea", "BsmtHalfBath", "BsmtFullBath", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "GarageCars"]
data[features_zerofill] = data[features_zerofill].fillna(0)

# 1. Identify any remaining missing values
missing_values = data.isna().sum()
missing_values = missing_values[missing_values > 0]

print("Remaining missing values in data DataFrame:")
print(missing_values)

# 2. Create lists of numerical and categorical column names
numerical_cols = data.select_dtypes(include=np.number).columns.tolist()
categorical_cols = data.select_dtypes(include='object').columns.tolist()

# Some numerical columns might be better treated as categorical if they represent discrete groups
# For example, 'MSSubClass' is often treated as categorical.
# Let's ensure 'Id' is not in numerical_cols if it was implicitly included.
if 'Id' in numerical_cols:
    numerical_cols.remove('Id')

# Handle columns that might have slipped through and are better as categorical
# For example, 'MoSold' and 'YrSold' are typically not scaled numerically
# Convert these to object type for the categorical pipeline if they exist
for col in ['MSSubClass', 'MoSold', 'YrSold']:
    if col in numerical_cols:
        numerical_cols.remove(col)
        data[col] = data[col].astype(str) # Convert to string to be treated as categorical
        categorical_cols.append(col)

# Re-evaluate categorical_cols based on the conversion, if any
categorical_cols = data.select_dtypes(include='object').columns.tolist()

print("\nNumerical Columns:", numerical_cols)
print("Categorical Columns:", categorical_cols)

# 3. Define a numeric_transformer pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# 4. Define a categorical_transformer pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # Explicitly set sparse_output=False
])

# 5. Create a ColumnTransformer named preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 6. Apply the preprocessor to the data DataFrame
data_preprocessed = preprocessor.fit_transform(data)

# 7. Convert the data_preprocessed output from a NumPy array back into a Pandas DataFrame
# Get feature names after one-hot encoding
feature_names = preprocessor.get_feature_names_out()
data_preprocessed = pd.DataFrame(data_preprocessed, columns=feature_names)

print("\nShape of preprocessed data:", data_preprocessed.shape)
print("Preprocessed data head:")
print(data_preprocessed.head())

#### Corelation Heatmaps

In [ ]:
sns.set(font_scale=1.1)
corr_train = train_data.corr(numeric_only=True)
mask = np.triu(corr_train) # Corrected: mask should be applied directly to corr_train
plt.figure(figsize=(20, 20))
ax = plt.gca() # Get current axes
ax.set_facecolor('white') # Set axes background to white
fig = plt.gcf() # Get current figure
fig.set_facecolor('white') # Set figure background to white
sns.heatmap(corr_train, annot=True, fmt='.1f', cmap='coolwarm', square=True, mask=mask, linewidth=0, cbar=True, ax=ax) # Set linewidth=0 for no lines between cells
plt.show()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# 1. Convert data_preprocessed DataFrame into a NumPy array
X_processed_array = data_preprocessed.to_numpy()

# Get the length of the original training data after outlier removal
# train_data was updated in cell FrSpIQ5qaERH: train_data = train_data[train_data["GrLivArea"]<4450]
# y was created in cell qroW6QSCaERL: y = np.log(train_data["SalePrice"])

# The current `train_data` still holds the post-outlier-removal data, so its length is correct.
original_train_len = train_data.shape[0]

# 2. Split the X_processed_array into training features and test features
X_train_processed = X_processed_array[:original_train_len]
X_test_processed = X_processed_array[original_train_len:]

# Ensure y is defined and correctly aligned
y = np.log(train_data["SalePrice"])

# 3. Convert X_train_processed, y, and X_test_processed into PyTorch tensors
X_train_processed_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1) # Ensure target is 2D for PyTorch models
X_test_processed_tensor = torch.tensor(X_test_processed, dtype=torch.float32)

print(f"Shape of X_train_processed_tensor: {X_train_processed_tensor.shape}")
print(f"Shape of y_tensor: {y_tensor.shape}")
print(f"Shape of X_test_processed_tensor: {X_test_processed_tensor.shape}")

# 4. Split the training features and target into training and validation sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_processed_tensor, y_tensor, test_size=0.2, random_state=42
)

print(f"Shape of X_train_split: {X_train_split.shape}")
print(f"Shape of y_train_split: {y_train_split.shape}")
print(f"Shape of X_val_split: {X_val_split.shape}")
print(f"Shape of y_val_split: {y_val_split.shape}")

# 5. Define a custom PyTorch Dataset class
class HouseDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature = self.features[idx]
        if self.labels is not None:
            label = self.labels[idx]
            return feature, label
        return feature

# 6. Create instances of HouseDataset for the training set and validation set
train_dataset = HouseDataset(X_train_split, y_train_split)
val_dataset = HouseDataset(X_val_split, y_val_split)
# Create a dataset for the test features as well
test_dataset = HouseDataset(X_test_processed_tensor)

print(f"Number of samples in train_dataset: {len(train_dataset)}")
print(f"Number of samples in val_dataset: {len(val_dataset)}")
print(f"Number of samples in test_dataset: {len(test_dataset)}")

# 7. Create DataLoader instances
batch_size = 64 # You can adjust this batch size
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of batches in train_loader: {len(train_loader)}")
print(f"Number of batches in val_loader: {len(val_loader)}")
print(f"Number of batches in test_loader: {len(test_loader)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.grid(False) # Disable the grid
ax.scatter(train_data["GrLivArea"], train_data["SalePrice"], c="#3f72af", zorder=3, alpha=0.9)
ax.axvline(4500, c="#112d4e", ls="--", zorder=2)
ax.set_xlabel("Ground living area (sq. ft)", labelpad=10)
ax.set_ylabel("Sale price ($)", labelpad=10)
ax.set_facecolor('white') # Set axes background to white
fig.set_facecolor('white') # Set figure background to white
plt.show()

## PyTorch Regression Model


We design and implement a neural network architecture using PyTorch's `nn.Module` for regression. The model should be flexible enough to handle the number of input features after encoding and output a single value (the predicted sale price).


In [ ]:
import torch.nn as nn

class RegressionModel(nn.Module):
    def __init__(self, input_features):
        super(RegressionModel, self).__init__()
        # Input layer (input_features to 128 neurons)
        self.fc1 = nn.Linear(input_features, 128)
        # Hidden layer 1 (128 to 64 neurons)
        self.fc2 = nn.Linear(128, 64)
        # Output layer (64 to 1 neuron for regression)
        self.fc3 = nn.Linear(64, 1)

        # Activation function
        self.relu = nn.ReLU()

    def forward(self, x):
        # Pass input through layers and activation functions
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        # The output layer typically does not have an activation function for regression
        # unless a specific range is desired (e.g., sigmoid for [0,1], tanh for [-1,1])
        # For general regression, a linear output is common.
        x = self.fc3(x)
        return x

print("RegressionModel class defined successfully.")

## Defining Loss Functions
Here we define the different loss functions to be compared.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# 1. Define Mean Absolute Error (MAE) loss function
mae_loss = nn.L1Loss()

# 2. Define Mean Squared Error (MSE) loss function
mse_loss = nn.MSELoss()

# 3. Implement a custom regression hinge-like loss function
def regression_hinge_loss(predictions, targets, epsilon=0.5):
    # Calculate the absolute difference between predictions and targets
    error = torch.abs(predictions - targets)
    # Penalize errors only when they exceed epsilon
    loss = F.relu(error - epsilon)
    # Return the mean of the loss across the batch
    return torch.mean(loss)

print("MAE, MSE, and custom regression hinge-like loss functions defined.")

## Train Model with MAE Loss


In [ ]:
import torch.optim as optim

# 1. Initialize an instance of the RegressionModel class
input_features = X_train_processed_tensor.shape[1]
model = RegressionModel(input_features)

# 2. Define the optimizer
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 3. Set the number of training epochs
n_epochs = 100

# 4. Create empty lists to store training losses and validation losses
training_losses = []
validation_losses = []

print(f"Starting training for {n_epochs} epochs...")

# 5. Implement a training loop
for epoch in range(n_epochs):
    # a. Set the model to training mode
    model.train()
    current_train_loss = 0.0

    # b. Iterate through the train_loader
    for batch_idx, (features, labels) in enumerate(train_loader):
        # c. For each batch:
        # i. Zero the optimizer's gradients
        optimizer.zero_grad()

        # ii. Perform a forward pass
        predictions = model(features)

        # iii. Calculate the MAE loss
        loss = mae_loss(predictions, labels)

        # iv. Perform a backward pass
        loss.backward()

        # v. Update the model's weights
        optimizer.step()

        current_train_loss += loss.item()

    # d. Calculate average training loss for the epoch
    avg_train_loss = current_train_loss / len(train_loader)
    training_losses.append(avg_train_loss)

    # e. Set the model to evaluation mode
    model.eval()
    current_val_loss = 0.0

    # f. Disable gradient calculation
    with torch.no_grad():
        # g. Iterate through the val_loader
        for features, labels in val_loader:
            # h. Perform a forward pass and calculate MAE loss
            predictions = model(features)
            loss = mae_loss(predictions, labels)
            current_val_loss += loss.item()

    # i. Calculate average validation loss for the epoch
    avg_val_loss = current_val_loss / len(val_loader)
    validation_losses.append(avg_val_loss)

    # j. Print the training and validation loss for the current epoch
    print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

print("Training finished.")

## Train Model with MSE Loss



In [ ]:
import torch.optim as optim

# 1. Initialize a new instance of the RegressionModel class for MSE training
mse_model = RegressionModel(input_features)

# 2. Define the optimizer for mse_model
learning_rate = 0.001
mse_optimizer = optim.Adam(mse_model.parameters(), lr=learning_rate)

# 3. Set the number of training epochs
n_epochs = 100

# 4. Create empty lists to store training losses and validation losses for MSE model
mse_training_losses = []
mse_validation_losses = []

print(f"Starting training for MSE model for {n_epochs} epochs...")

# 5. Implement a training loop for the MSE model
for epoch in range(n_epochs):
    # a. Set the model to training mode
    mse_model.train()
    current_train_loss_mse = 0.0

    # b. Iterate through the train_loader
    for batch_idx, (features, labels) in enumerate(train_loader):
        # c. For each batch:
        # i. Zero the optimizer's gradients
        mse_optimizer.zero_grad()

        # ii. Perform a forward pass
        predictions = mse_model(features)

        # iii. Calculate the MSE loss
        loss = mse_loss(predictions, labels)

        # iv. Perform a backward pass
        loss.backward()

        # v. Update the model's weights
        mse_optimizer.step()

        current_train_loss_mse += loss.item()

    # e. Calculate average training loss for the epoch
    avg_train_loss_mse = current_train_loss_mse / len(train_loader)
    mse_training_losses.append(avg_train_loss_mse)

    # f. Set the model to evaluation mode
    mse_model.eval()
    current_val_loss_mse = 0.0

    # g. Disable gradient calculation
    with torch.no_grad():
        # h. Iterate through the val_loader
        for features, labels in val_loader:
            # i. Perform a forward pass and calculate MSE loss
            predictions = mse_model(features)
            loss = mse_loss(predictions, labels)
            current_val_loss_mse += loss.item()

    # j. Calculate average validation loss for the epoch
    avg_val_loss_mse = current_val_loss_mse / len(val_loader)
    mse_validation_losses.append(avg_val_loss_mse)

    # k. Print the training and validation loss for the current epoch
    print(f"Epoch {epoch+1}/{n_epochs}, MSE Train Loss: {avg_train_loss_mse:.4f}, MSE Val Loss: {avg_val_loss_mse:.4f}")

print("MSE model training finished.")

## Train Model with Hinge Loss


In [ ]:
import torch.optim as optim

# 1. Initialize a new instance of the RegressionModel class for Hinge loss training
hinge_model = RegressionModel(input_features)

# 2. Define the optimizer for hinge_model
learning_rate = 0.001
hinge_optimizer = optim.Adam(hinge_model.parameters(), lr=learning_rate)

# 3. Set the number of training epochs
n_epochs = 100

# 4. Create empty lists to store training losses and validation losses for Hinge model
hinge_training_losses = []
hinge_validation_losses = []

print(f"Starting training for Hinge model for {n_epochs} epochs...")

# 5. Implement a training loop for the Hinge model
for epoch in range(n_epochs):
    # a. Set the model to training mode
    hinge_model.train()
    current_train_loss_hinge = 0.0

    # b. Iterate through the train_loader
    for batch_idx, (features, labels) in enumerate(train_loader):
        # c. For each batch:
        # i. Zero the optimizer's gradients
        hinge_optimizer.zero_grad()

        # ii. Perform a forward pass
        predictions = hinge_model(features)

        # iii. Calculate the hinge loss
        loss = regression_hinge_loss(predictions, labels) # Using the custom defined hinge loss

        # iv. Perform a backward pass
        loss.backward()

        # v. Update the model's weights
        hinge_optimizer.step()

        current_train_loss_hinge += loss.item()

    # d. Calculate average training loss for the epoch
    avg_train_loss_hinge = current_train_loss_hinge / len(train_loader)
    hinge_training_losses.append(avg_train_loss_hinge)

    # e. Set the model to evaluation mode
    hinge_model.eval()
    current_val_loss_hinge = 0.0

    # f. Disable gradient calculation
    with torch.no_grad():
        # g. Iterate through the val_loader
        for features, labels in val_loader:
            # h. Perform a forward pass and calculate Hinge loss
            predictions = hinge_model(features)
            loss = regression_hinge_loss(predictions, labels)
            current_val_loss_hinge += loss.item()

    # i. Calculate average validation loss for the epoch
    avg_val_loss_hinge = current_val_loss_hinge / len(val_loader)
    hinge_validation_losses.append(avg_val_loss_hinge)

    # j. Print the training and validation loss for the current epoch
    print(f"Epoch {epoch+1}/{n_epochs}, Hinge Train Loss: {avg_train_loss_hinge:.4f}, Hinge Val Loss: {avg_val_loss_hinge:.4f}")

print("Hinge model training finished.")

## Evaluate Models and Visualize Results
Now for each trained model (MAE, MSE, Hinge), we calculate the R-squared score on the test set. Then, we Generate plots for each model showing: 1) training and validation loss curves over epochs, and 2) a scatter plot of actual vs. predicted `SalePrice` values.


In [ ]:
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

print("r2_score and matplotlib.pyplot imported.")

In [ ]:
print("\n--- Evaluating MAE Model ---")

# 2a. Set the MAE model to evaluation mode
model.eval()

# 2b & 2c. Make predictions on the validation set and convert to numpy
with torch.no_grad():
    mae_val_predictions = model(X_val_split).detach().numpy()

# 2e. Apply np.exp() to predictions and actual validation labels
mae_val_predictions_exp = np.exp(mae_val_predictions)
y_val_split_exp = np.exp(y_val_split.numpy())

# 2e & 2f. Calculate and print R-squared score on the validation set
mae_r2 = r2_score(y_val_split_exp, mae_val_predictions_exp)
print(f"MAE Model - R-squared on Validation Set: {mae_r2:.4f}")

# 2g. Plot training and validation loss curves for MAE model
plt.figure(figsize=(10, 5))
ax_loss = plt.gca() # Get current axes
ax_loss.set_facecolor('white') # Set axes background to white
fig_loss = plt.gcf() # Get current figure
fig_loss.set_facecolor('white') # Set figure background to white
plt.plot(training_losses, label='Training Loss')
plt.plot(validation_losses, label='Validation Loss')
plt.title('MAE Loss - Training and Validation Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)
plt.show()

# 2h. Create a scatter plot of actual vs. predicted values for MAE model (validation set)
plt.figure(figsize=(10, 5))
ax_scatter = plt.gca() # Get current axes
ax_scatter.set_facecolor('white') # Set axes background to white
fig_scatter = plt.gcf() # Get current figure
fig_scatter.set_facecolor('white') # Set figure background to white
plt.scatter(y_val_split_exp, mae_val_predictions_exp, alpha=0.6)
plt.plot([y_val_split_exp.min(), y_val_split_exp.max()], [y_val_split_exp.min(), y_val_split_exp.max()], 'r--', lw=2, label='Perfect Prediction')
plt.title('MAE Model - Actual vs. Predicted Sale Price (Validation Set)')
plt.xlabel('Actual Sale Price')
plt.ylabel('Predicted Sale Price')
plt.legend()
plt.grid(False)
plt.show()

In [ ]:
print("\n--- Evaluating MSE Model ---")

# 2a. Set the MSE model to evaluation mode
mse_model.eval()

# 2b & 2c. Make predictions on the validation set and convert to numpy
with torch.no_grad():
    mse_val_predictions = mse_model(X_val_split).detach().numpy()

# 2e. Apply np.exp() to predictions and actual validation labels
mse_val_predictions_exp = np.exp(mse_val_predictions)
# y_val_split_exp is already calculated from the previous step

# 2e & 2f. Calculate and print R-squared score on the validation set
mse_r2 = r2_score(y_val_split_exp, mse_val_predictions_exp)
print(f"MSE Model - R-squared on Validation Set: {mse_r2:.4f}")

# 2g. Plot training and validation loss curves for MSE model
plt.figure(figsize=(10, 5))
ax_loss = plt.gca() # Get current axes
ax_loss.set_facecolor('white') # Set axes background to white
fig_loss = plt.gcf() # Get current figure
fig_loss.set_facecolor('white') # Set figure background to white
plt.plot(mse_training_losses, label='Training Loss')
plt.plot(mse_validation_losses, label='Validation Loss')
plt.title('MSE Loss - Training and Validation Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)
plt.show()

# 2h. Create a scatter plot of actual vs. predicted values for MSE model (validation set)
plt.figure(figsize=(10, 5))
ax_scatter = plt.gca() # Get current axes
ax_scatter.set_facecolor('white') # Set axes background to white
fig_scatter = plt.gcf() # Get current figure
fig_scatter.set_facecolor('white') # Set figure background to white
plt.scatter(y_val_split_exp, mse_val_predictions_exp, alpha=0.6)
plt.plot([y_val_split_exp.min(), y_val_split_exp.max()], [y_val_split_exp.min(), y_val_split_exp.max()], 'r--', lw=2, label='Perfect Prediction')
plt.title('MSE Model - Actual vs. Predicted Sale Price (Validation Set)')
plt.xlabel('Actual Sale Price')
plt.ylabel('Predicted Sale Price')
plt.legend()
plt.grid(False)
plt.show()

In [ ]:
print("\n--- Evaluating Hinge Model ---")

# 2a. Set the Hinge model to evaluation mode
hinge_model.eval()

# 2b & 2c. Make predictions on the validation set and convert to numpy
with torch.no_grad():
    hinge_val_predictions = hinge_model(X_val_split).detach().numpy()

# 2e. Apply np.exp() to predictions and actual validation labels
hinge_val_predictions_exp = np.exp(hinge_val_predictions)
# y_val_split_exp is already calculated from the previous step

# 2e & 2f. Calculate and print R-squared score on the validation set
hinge_r2 = r2_score(y_val_split_exp, hinge_val_predictions_exp)
print(f"Hinge Model - R-squared on Validation Set: {hinge_r2:.4f}")

# 2g. Plot training and validation loss curves for Hinge model
plt.figure(figsize=(10, 5))
ax_loss = plt.gca() # Get current axes
ax_loss.set_facecolor('white') # Set axes background to white
fig_loss = plt.gcf() # Get current figure
fig_loss.set_facecolor('white') # Set figure background to white
plt.plot(hinge_training_losses, label='Training Loss')
plt.plot(hinge_validation_losses, label='Validation Loss')
plt.title('Hinge Loss - Training and Validation Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)
plt.show()

# 2h. Create a scatter plot of actual vs. predicted values for Hinge model (validation set)
plt.figure(figsize=(10, 5))
ax_scatter = plt.gca() # Get current axes
ax_scatter.set_facecolor('white') # Set axes background to white
fig_scatter = plt.gcf() # Get current figure
fig_scatter.set_facecolor('white') # Set figure background to white
plt.scatter(y_val_split_exp, hinge_val_predictions_exp, alpha=0.6)
plt.plot([y_val_split_exp.min(), y_val_split_exp.max()], [y_val_split_exp.min(), y_val_split_exp.max()], 'r--', lw=2, label='Perfect Prediction')
plt.title('Hinge Model - Actual vs. Predicted Sale Price (Validation Set)')
plt.xlabel('Actual Sale Price')
plt.ylabel('Predicted Sale Price')
plt.legend()
plt.grid(False)
plt.show()

## Notes to ponder:

Which loss function performed best?

Why did it perform best?